# APMC catalog -- initial extraction (chunk vision)

This notebook produces the initial 4-column CSV (`Listing`, `Page`, `Images Above`,
`Type`) for an OCR'd ASCC catalog by sending each chunk PNG to a Claude vision
model and asking for the entries in top-to-bottom order. A chunk is either pure
catalog text, or a marking-image section at the top followed by catalog text.
Each entry is classified as `LISTING` (a catalog entry) or `META` (section
header, running head, page number, cross-reference, fragment, etc.). The output
is the upstream input that `apmc_data_munger.ipynb` consumes.

Inputs:  `wip/in/<BASENAME>/page-NNNN-MMMM.png`  (page, chunk_seq)
Output:  `wip/in/<BASENAME>.csv`
Cache:   `wip/cache/<BASENAME>_extract.json`

Stages:

1. Setup and config.
2. Discover the chunk PNGs.
3. System prompt (frozen string, prompt-cached).
4. Vision call function.
5. Cache + loop -- one call per chunk.
6. Assemble the DataFrame.
7. Write the CSV.
8. Verification notes.


In [ ]:
# 1. Setup and config.
import base64, json, os, re
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv
import anthropic

# Repo-root .env (notebook cwd is tools/).
load_dotenv(Path("..") / ".env")

pd.set_option("display.max_colwidth", 200)
pd.set_option("display.width", 200)

BASENAME   = "VA_ASCC_CTLG"
IMAGES_DIR = Path(f"./wip/in/{BASENAME}")
OUTPUT_CSV = Path(f"./wip/in/{BASENAME}.csv")
CACHE_FILE = Path(f"./wip/cache/{BASENAME}_extract.json")
CACHE_FILE.parent.mkdir(parents=True, exist_ok=True)
OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)

EXTRACT_MODEL          = "claude-sonnet-4-6"
EXTRACT_PROMPT_VERSION = "v2"
EXTRACT_MAX_TOKENS     = 8192   # high-density chunks can carry 40+ entries; 4096 truncated
FORCE_REFRESH          = False

assert IMAGES_DIR.is_dir(), f"missing images dir: {IMAGES_DIR}"
assert os.environ.get("ANTHROPIC_API_KEY"), "ANTHROPIC_API_KEY not set"
client = anthropic.Anthropic()
print(f"model: {EXTRACT_MODEL}  prompt_version: {EXTRACT_PROMPT_VERSION}")
print(f"images: {IMAGES_DIR}")
print(f"output: {OUTPUT_CSV}")
print(f"cache:  {CACHE_FILE}")


In [ ]:
# 2. Discover the chunk PNGs and sort into reading order.
NAME_RE = re.compile(r"^page-(\d{4})-(\d{4})\.png$")

chunks = []
for p in sorted(IMAGES_DIR.glob("page-*-*.png")):
    m = NAME_RE.match(p.name)
    if not m:
        continue
    chunks.append((int(m.group(1)), int(m.group(2)), p))

# Reading order: ascending page, then ascending chunk_seq.
chunks.sort(key=lambda t: (t[0], t[1]))

pages = sorted({c[0] for c in chunks})
assert chunks, f"no PNGs matched in {IMAGES_DIR}"
print(f"found {len(chunks)} chunks across pages {pages[0]}-{pages[-1]} "
      f"({len(pages)} pages total)")


## 3. System prompt

Frozen string sent with `cache_control: ephemeral` so prompt caching hits on
calls within the 5-minute TTL. The prompt instructs the model to return strict
JSON only and shows the schema by example.


In [ ]:
# 3. System prompt for the extraction model.
EXTRACT_SYSTEM_PROMPT = """You are reading chunk scans of an old American Stampless Cover (ASCC) catalog.

Each image is one chunk of an ASCC catalog page. A chunk is either pure
catalog text, or a marking-image section at the top followed by catalog
text. The remainder of the chunk (below any top image section) is text,
read top-to-bottom.

Your job: emit every piece of text visible in the chunk, in top-to-bottom
order, as strict JSON. For each text entry, classify it as LISTING (a
catalog entry) or META (anything else). Also return ONE chunk-level integer
count of distinct marking-image reproductions in the top image section.

What is a LISTING vs META
-------------------------

A LISTING is a single catalog entry. Classify an entry as LISTING if its
text has ANY of these signals:

- trailing valuation: ends with a number (e.g. 1500.00) or with -- / ---
- semicolon-parenthetical of the form (...; ...; ...)
- a 4-digit year matching 17xx or 18xx
- a relationship-indicator prefix: the literal word Same, or (L), or (E)

Examples of LISTING text drawn verbatim from this catalog:

    Alexa.(Alexandria)(E)(May 21,1772;Ms;Black) ......... 1500.00
    Fredg.(Aug.2,1772;Ms;Black) .. 1500.00
    (L)(Sept.15,1774) . . .. 1000.00
    Same(Aug.2,1772;Ms;Black) .. 1500.00

Continuation lines that wrap an entry onto a second line (begin with an
open paren but not the word Same) are part of the SAME listing -- join the
wrapped text into one LISTING string.

A continuation listing that begins with the literal word Same followed by
an open paren is a SEPARATE LISTING. Emit each Same(...) line as its own
LISTING entry.

Classify as META anything that is not a LISTING. Examples:

- section headings (e.g. VIRGINIA)
- running heads / column headers
- printed page numbers
- editorial paragraphs and notes
- cross-references (e.g. "See Alexandria") with no semicolon-parenthetical
- isolated fragments
- any other non-entry text

Skip the marking-image reproductions themselves -- they are counted in
images_above, not emitted as entries.

Text fidelity (LISTING and META)
--------------------------------

Reproduce the text EXACTLY as printed. Preserve case, spacing, punctuation,
semicolons, slashes, parentheses, leader dots, and any trailing price
value. Do NOT parse, split, normalize, expand abbreviations, or "clean up"
leader dots.

ASCII only. Use straight " and ' (not curly), use - or -- (not en-dash or
em-dash), use three dots ... (not the ellipsis character), no accented
letters, no Unicode bullets or arrows.

If a piece of text is fully illegible, OMIT it. Do not fabricate text. If
only part is illegible, emit what you can read faithfully and use ? in
place of the unreadable glyphs.

Counting images_above
---------------------

Return ONE integer for the whole chunk:

- If the chunk has a marking-image section at the top, count the distinct
  marking-image reproductions in that top section.
- If the chunk is pure text (no top image section), return 0.

Rules:

- ONE stamp image is ONE image. A small adjacent date or rate annotation
  printed beside the stamp is part of the same image, NOT a second image.
- Two genuinely separate stamp reproductions placed close together are
  TWO images.
- If unsure between 1 and 2, prefer 1.

Output format
-------------

Return STRICT JSON only -- no surrounding prose, no markdown fences, no
trailing commentary. The shape:

    {
      "images_above": 0,
      "entries": [
        {"text": "Alexa.(Alexandria)(E)(May 21,1772;Ms;Black) ......... 1500.00", "type": "LISTING"},
        {"text": "VIRGINIA", "type": "META"}
      ]
    }

If the chunk has no extractable text, return {"images_above": 0, "entries": []}.
"""

print(f"system prompt length: {len(EXTRACT_SYSTEM_PROMPT)} chars")


In [ ]:
# 4. Extraction call. Returns the parsed {images_above, entries} dict.

def _strip_fences(text: str) -> str:
    # Some models wrap JSON in ```json ... ``` fences despite instructions.
    t = text.strip()
    if t.startswith("```"):
        # drop opening fence line
        t = t.split("\n", 1)[1] if "\n" in t else t[3:]
        # drop trailing fence
        if t.rstrip().endswith("```"):
            t = t.rstrip()[:-3]
    return t.strip()

def _call_model(image_b64: str, user_prompt: str) -> str:
    msg = client.messages.create(
        model=EXTRACT_MODEL,
        max_tokens=EXTRACT_MAX_TOKENS,
        system=[{
            "type": "text",
            "text": EXTRACT_SYSTEM_PROMPT,
            "cache_control": {"type": "ephemeral"},
        }],
        messages=[{"role": "user", "content": [
            {"type": "image", "source": {
                "type": "base64",
                "media_type": "image/png",
                "data": image_b64,
            }},
            {"type": "text", "text": user_prompt},
        ]}],
    )
    text_blocks = [b.text for b in msg.content if b.type == "text"]
    return text_blocks[0] if text_blocks else ""

def extract_chunk(image_path: Path, page: int, chunk_seq: int) -> dict:
    user_prompt = (
        f"This image is chunk {chunk_seq} of ASCC catalog page {page}. "
        f"Return strict JSON of the form "
        f'{{"images_above": 0, "entries": [{{"text": "...", "type": "LISTING"|"META"}}, ...]}}.'
    )
    image_b64 = base64.standard_b64encode(image_path.read_bytes()).decode()

    last_raw = ""
    for attempt in range(2):
        raw = _call_model(image_b64, user_prompt)
        last_raw = raw
        cleaned = _strip_fences(raw)
        if not cleaned:
            continue
        try:
            parsed = json.loads(cleaned)
            break
        except json.JSONDecodeError:
            continue
    else:
        snippet = (last_raw[:200] + "...") if len(last_raw) > 200 else last_raw
        raise ValueError(f"model returned non-JSON after 2 attempts; raw={snippet!r}")

    assert isinstance(parsed, dict), f"top-level not dict: {type(parsed)}"
    images_above = parsed.get("images_above")
    assert isinstance(images_above, int) and images_above >= 0, \
        f"images_above bad: {images_above!r}"
    entries = parsed.get("entries")
    assert isinstance(entries, list), f"entries not list: {type(entries)}"
    for i, r in enumerate(entries):
        assert isinstance(r, dict), f"entry[{i}] not dict"
        assert isinstance(r.get("text"), str), f"entry[{i}].text not str"
        assert r.get("type") in ("LISTING", "META"), \
            f"entry[{i}].type bad: {r.get('type')!r}"
    return {"images_above": images_above, "entries": entries}


## 5. Cache and loop

Disk cache is invalidated when `EXTRACT_MODEL` or `EXTRACT_PROMPT_VERSION`
changes. Cache is saved after each successful call so a crash mid-loop only
loses the in-flight call. Set `FORCE_REFRESH = True` in the config cell to
re-query everything.


In [ ]:
# 5. Cache + loop. One call per chunk.

def load_cache(path: Path) -> dict:
    if not path.exists():
        return {"model": EXTRACT_MODEL, "prompt_version": EXTRACT_PROMPT_VERSION, "responses": {}}
    cache = json.loads(path.read_text())
    if cache.get("model") != EXTRACT_MODEL or cache.get("prompt_version") != EXTRACT_PROMPT_VERSION:
        print(f"cache invalidated (was model={cache.get('model')!r}, prompt={cache.get('prompt_version')!r})")
        return {"model": EXTRACT_MODEL, "prompt_version": EXTRACT_PROMPT_VERSION, "responses": {}}
    return cache

def save_cache(path: Path, cache: dict) -> None:
    path.write_text(json.dumps(cache, indent=2))

cache = load_cache(CACHE_FILE)
responses = cache["responses"]

calls_made = 0
for page, chunk_seq, img_path in chunks:
    key = f"page-{page:04d}-{chunk_seq:04d}"
    if key in responses and not FORCE_REFRESH:
        continue
    if not img_path.exists():
        print(f"missing image, skipping: {img_path}")
        continue
    try:
        result = extract_chunk(img_path, page, chunk_seq)
    except Exception as e:
        print(f"  {key}: FAILED ({type(e).__name__}: {e})")
        save_cache(CACHE_FILE, cache)
        raise
    responses[key] = result
    save_cache(CACHE_FILE, cache)
    calls_made += 1
    n = len(result["entries"])
    print(f"  {key}: {n:3d} entries, images_above={result['images_above']}")

print()
print(f"calls made: {calls_made}")
print(f"cached responses: {len(responses)}")


## 6. Assemble the DataFrame

Walk the chunk list in catalog reading order, pull each chunk's entries from
the cache, and append rows. The chunk's single `images_above` count is
attached to the FIRST entry of that chunk; every other entry in the chunk
gets `Images Above = 0`. The `Page` column is filled in from the parsed
filename `NNNN`.


In [ ]:
# 6. Assemble DataFrame from the cached responses.
rows = []
for page, chunk_seq, _ in chunks:
    key = f"page-{page:04d}-{chunk_seq:04d}"
    entry = responses.get(key)
    if entry is None:
        print(f"WARNING: no cached response for {key}; skipping")
        continue
    chunk_images = int(entry["images_above"])
    for i, r in enumerate(entry["entries"]):
        rows.append({
            "Listing": r["text"],
            "Page": int(page),
            "Images Above": chunk_images if i == 0 else 0,
            "Type": r["type"],
        })

df = pd.DataFrame(rows, columns=["Listing", "Page", "Images Above", "Type"])
print(f"rows: {len(df):,}")
print(f"pages: {df.Page.min()}-{df.Page.max()}")
print()
print("rows per page:")
print(df.groupby("Page").size().rename("rows").to_frame())
print()
print("Images Above value counts:")
print(df["Images Above"].value_counts().sort_index().rename("rows").to_frame())
print()
print("Type value counts:")
print(df["Type"].value_counts().rename("rows").to_frame())
df.head(5)


## 7. Write the CSV

Pandas' `to_csv` default is RFC 4180 compliant: UTF-8, no BOM, header row,
fields containing commas or quotes are double-quoted. The output file is
the munger's input.


In [ ]:
# 7. Write CSV.
df[["Listing", "Page", "Images Above", "Type"]].to_csv(OUTPUT_CSV, index=False)
print(f"wrote: {OUTPUT_CSV}")
print(f"rows:  {len(df):,}")
print(f"pages: {df.Page.min()}-{df.Page.max()}")


## 8. Verification

* First run hits the API once per chunk. Re-running cell 5 should print
  `calls made: 0` (everything served from `wip/cache/<BASENAME>_extract.json`).
* Spot-check: open `wip/in/<BASENAME>/page-NNNN-MMMM.png` next to
  `df.query("Page == NNNN").head(20)`. Entries should appear in the same
  top-to-bottom order, glyph-for-glyph match on the text, the FIRST entry
  of each chunk should carry the chunk-level `Images Above` count, and
  every entry should be classified `LISTING` or `META`.
* The output CSV at `wip/in/<BASENAME>.csv` is the direct input to
  `apmc_data_munger.ipynb`.
